# 80 embed

Question: does `aosabi_core::embed::answer`, the entry of a host that links the engine and the roots
of its app into its own program, answer the JSON body of a GraphQL request with the bytes a request
of the same document answers through the service, and does it answer a body that is no request with
one error and no data.

The side under test is the operation `embed` of `aosabi-devtool execute`, which hands a body to the
service of a loaded behaviour table through the entry. The reference side is the operation
`execute` of the same primitive over the same table, the executor that the served binaries and the
mode `--requests` reach, so the notebook holds the new entry to an answer the conformance suite
judges already. The corpus is the one of `tests/capture/20_corpus/`, every table of the mode
`pylon`, which is the mode whose tables become a service.

The section The facade of a host then judges the facade a host links, `aosabi::embed::Embedded`,
through the program `tests/fixtures/rs/embed-host` over the app `tests/fixtures/rs/embed-app`. Its
reference side is the binary the card `server` builds for the same app, in the mode `--requests`,
and for a body that is no request the operation `embed` of the devtool. The section holds the host
to the key `maskedErrors` of its app as well, which masks an unexpected error only where the app
sets it, to the error of the boot of an app whose key sets a mask function of its own, and to the
key `introspection`, which refuses introspection only where the app sets it to `false`.

The section The key of a fresh thread holds every entry that answers a request to keying the
thread that calls it before it reads the request, and the section A target without random bytes
holds the entry, on `wasm32-unknown-unknown`, to its refusal of every request until the host
installs a provider of random bytes, and to its answer once it has.

The section The nesting bound of a request holds the entry to the bound of 1,000 levels under
which the service parses every request document, and the section The token cap of a request to the
cap of 120,008 tokens under which it parses each. The section The key of the token cap holds the
facade of a host and the binary of the card `server` to the key `maxTokens` of the app, set lower
and higher in its chain.

In [ ]:
# Papermill overrides this cell.
# The tree under comparison.
aosabi_root = "."

# The per command timeout.
timeout_s = 600

# The pinned toolchain the crate is built with.
rust_version = "1.95.0"

## Setup

The cell prints the toolchain and the gates the notebook rests on: the builds of the devtool and of
the command line, the lints of the workspace, and the format of both files of the entry, of the
devtool and of the sources of both hosts and their app, every cargo call with `--locked`.

In [ ]:
import json
import os
import sys
from pathlib import Path

for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "tests" / "harness" / "__init__.py").is_file():
        sys.path.insert(0, str(_base / "tests"))
        break

from harness import GoldenMissing, bytediff, golden_read, lock_dir, paths, run
from harness.fixtures import build_env, rustup_cargo

P = paths(aosabi_root)
OUT = P.out / "80_embed"
OUT.mkdir(parents=True, exist_ok=True)
DEVTOOL = P.target / "debug" / "aosabi-devtool"
CARGO_BIN = Path(os.environ.get("CARGO_HOME") or Path.home() / ".cargo") / "bin"
CARGO = CARGO_BIN / "cargo"
RUSTFMT = CARGO_BIN / "rustfmt"
CORPUS = P.tests / "capture" / "20_corpus"
CARGO_AOSABI = P.target / "debug" / "cargo-aosabi"
APP = P.tests / "fixtures" / "rs" / "embed-app"
HOST = P.tests / "fixtures" / "rs" / "embed-host"
WASM_HOST = P.tests / "fixtures" / "rs" / "embed-wasm"

# The gates this notebook rests on, each cargo call with --locked: the builds
# of the seam it drives and of the command line that builds the app of the
# host, the lints of the workspace, and the format of the two files of the
# entry, of the devtool and of the sources of both hosts and their app, which
# rustfmt checks because cargo fmt takes no --locked and none of those crates
# is a member of the workspace.
GATES = [
    ("cargo build --locked -p aosabi-devtool",
     [CARGO, "build", "--locked", "-p", "aosabi-devtool"]),
    ("cargo build --locked -p aosabi-cli --bin cargo-aosabi",
     [CARGO, "build", "--locked", "-p", "aosabi-cli", "--bin", "cargo-aosabi"]),
    ("cargo clippy --locked --workspace --all-targets -- -D warnings",
     [CARGO, "clippy", "--locked", "--workspace", "--all-targets", "--", "-D", "warnings"]),
    ("rustfmt --check --edition 2024 crates/aosabi-core/src/embed.rs",
     [RUSTFMT, "--check", "--edition", "2024", P.root / "crates" / "aosabi-core" / "src" / "embed.rs"]),
    ("rustfmt --check --edition 2024 crates/aosabi/src/embed.rs",
     [RUSTFMT, "--check", "--edition", "2024", P.root / "crates" / "aosabi" / "src" / "embed.rs"]),
    ("rustfmt --check --edition 2024 crates/aosabi-devtool/src/main.rs",
     [RUSTFMT, "--check", "--edition", "2024", P.root / "crates" / "aosabi-devtool" / "src" / "main.rs"]),
    ("rustfmt --check --edition 2024 tests/fixtures/rs/embed-app/src/lib.rs",
     [RUSTFMT, "--check", "--edition", "2024", APP / "src" / "lib.rs"]),
    ("rustfmt --check --edition 2024 tests/fixtures/rs/embed-host/build.rs",
     [RUSTFMT, "--check", "--edition", "2024", HOST / "build.rs"]),
    ("rustfmt --check --edition 2024 tests/fixtures/rs/embed-host/src/main.rs",
     [RUSTFMT, "--check", "--edition", "2024", HOST / "src" / "main.rs"]),
    ("rustfmt --check --edition 2024 tests/fixtures/rs/embed-wasm/build.rs",
     [RUSTFMT, "--check", "--edition", "2024", WASM_HOST / "build.rs"]),
    ("rustfmt --check --edition 2024 tests/fixtures/rs/embed-wasm/src/main.rs",
     [RUSTFMT, "--check", "--edition", "2024", WASM_HOST / "src" / "main.rs"]),
]
GATE_FAILURES = []
for label, argv in GATES:
    gate = run(argv, cwd=P.root, timeout_s=900, check=False)
    print("gate %-72s exit %d, %.1f s" % (label, gate.returncode, gate.duration_s))
    if gate.returncode != 0:
        GATE_FAILURES.append(label)
        print((gate.err or gate.out).strip()[-2000:])

reported_cargo = run([CARGO, "--version"], cwd=P.root, timeout_s=timeout_s).out.strip()
smoke = json.loads(run([DEVTOOL, "execute", "--smoke"], cwd=P.root, timeout_s=timeout_s).out)
print("repository root:", P.rel(P.root))
print("devtool        :", P.rel(DEVTOOL))
print("corpus         :", P.rel(CORPUS))
print("cargo          :", reported_cargo)
print("seam           :", json.dumps({key: smoke.get(key) for key in ("subcommand", "implemented")}))
assert not GATE_FAILURES, GATE_FAILURES
assert rust_version in reported_cargo, reported_cargo
assert smoke.get("subcommand") == "execute" and smoke.get("implemented") is True, smoke

DIFFS = {}
# The cases this checkout cannot judge, each with the place it needs, which the
# section and the verdict print where a place is absent.
NOT_JUDGED = []


def batch(requests, name):
    """Send `requests` to `execute` in one process and return its answers by id."""
    path = OUT / ("%s.ndjson" % name)
    path.write_text("".join(json.dumps(r) + "\n" for r in requests), encoding="utf-8")
    result = run([DEVTOOL, "execute", "--requests", path], cwd=P.root, timeout_s=timeout_s)
    (OUT / ("%s.answers.ndjson" % name)).write_text(result.out, encoding="utf-8")
    answers = {}
    for line in result.out.splitlines():
        answer = json.loads(line)
        answers[answer["id"]] = answer
    assert len(answers) == len(requests), (len(answers), len(requests))
    return answers


def body_of(case):
    """Returns the JSON body a client sends over HTTP for the request of `case`."""
    body = {"query": case["document"]}
    if case.get("variables") is not None:
        body["variables"] = case["variables"]
    if case.get("operationName") is not None:
        body["operationName"] = case["operationName"]
    return json.dumps(body, ensure_ascii=False)


def failure(message):
    """Returns the response body with the one error `message` and no data."""
    return json.dumps({"errors": [{"message": message}]}, ensure_ascii=False, separators=(",", ":"))

## The corpus through both entries

Every table of the mode `pylon` loads once, and every case of its corpus runs twice in one process,
as an `execute` of its document and as an `embed` of the body a client sends for it. The entry
validates every request, as a server does, so a case that turns validation off has no body that
asks for it and is counted and left out. Where the executor answers a result, the body is the
result text byte for byte. Where the resolver throws a value that is no GraphQL error, the executor
answers `thrown` and the body is the one error with the message of the thrown error, and the fixed
message for a thrown value that is no error.

In [ ]:
CORPORA = sorted(CORPUS.glob("*-pylon.json"))
compared = 0
skipped_unvalidated = 0
thrown_cases = 0
by_corpus = {}
for corpus_path in CORPORA:
    corpus = json.loads(corpus_path.read_text(encoding="utf-8"))
    requests = []
    for index, table in enumerate(corpus["tables"]):
        requests.append({"id": "t%d" % index, "op": "table", "table": table})
    pairs = []
    for index, case in enumerate(corpus["cases"]):
        if case.get("validate", True) is False:
            skipped_unvalidated += 1
            continue
        request = {"id": "x%d" % index, "op": "execute", "table": case["table"], "document": case["document"]}
        for key in ("variables", "operationName"):
            if case.get(key) is not None:
                request[key] = case[key]
        requests.append(request)
        requests.append({"id": "e%d" % index, "op": "embed", "table": case["table"], "body": body_of(case)})
        pairs.append(index)
    answers = batch(requests, corpus_path.stem)
    for index in range(len(corpus["tables"])):
        loaded = answers["t%d" % index]
        assert loaded.get("ok") is True or "errors" in loaded, loaded
    differing = 0
    for index in pairs:
        executed = answers["x%d" % index]
        embedded = answers["e%d" % index]
        assert "fatal" not in executed and "fatal" not in embedded, (executed, embedded)
        if "result" in executed:
            expected = executed["result"]
        else:
            thrown_cases += 1
            thrown = executed["thrown"]
            message = thrown.get("message")
            if thrown.get("name") is None or message is None:
                message = "the request threw a value that is no error"
            expected = failure(message)
        if expected != embedded["body"]:
            differing += 1
            if differing <= 3:
                bytediff(expected.encode(), embedded["body"].encode(), "%s case %d" % (corpus_path.stem, index))
        compared += 1
    by_corpus[corpus_path.stem] = (len(pairs), differing)
    print("%-28s %4d cases, %d differ" % (corpus_path.stem, len(pairs), differing))
DIFFS["corpus"] = sum(d for _, d in by_corpus.values())
print("compared:", compared, "of them thrown:", thrown_cases, "left out without validation:", skipped_unvalidated)
assert compared > 0

## Bodies that are no request

A body that is not JSON, that is no object, that holds no string `query` or whose `operationName`
is no string answers one error and no data. A `variables` or an `operationName` of `null` reads as
an absent one, as a GraphQL server over HTTP reads it.

In [ ]:
table = json.loads((CORPUS / "blog-app-pylon.json").read_text(encoding="utf-8"))["tables"][0]
name = table["name"]
query = next(c["document"] for c in json.loads((CORPUS / "blog-app-pylon.json").read_text(encoding="utf-8"))["cases"]
             if c["table"] == name and c.get("variables") is None and c.get("operationName") is None
             and c.get("validate", True))
BODIES = [
    ("not json", "{ broken", None),
    ("an array", "[1]", failure("the request body is not a JSON object")),
    ("a string", json.dumps("query"), failure("the request body is not a JSON object")),
    ("no query", "{}", failure("the request carries no string field query")),
    ("a query that is no string", json.dumps({"query": 3}), failure("the request carries no string field query")),
    ("an operationName that is no string", json.dumps({"query": query, "operationName": 3}),
     failure("the field operationName is no string")),
]
requests = [{"id": "t", "op": "table", "table": table}]
for index, (_, body, _) in enumerate(BODIES):
    requests.append({"id": "b%d" % index, "op": "embed", "table": name, "body": body})
plain = {"id": "n0", "op": "embed", "table": name, "body": json.dumps({"query": query})}
nulls = {"id": "n1", "op": "embed", "table": name,
         "body": json.dumps({"query": query, "variables": None, "operationName": None})}
requests += [plain, nulls]
answers = batch(requests, "bodies")
assert answers["t"].get("ok") is True, answers["t"]
differing = 0
for index, (label, body, expected) in enumerate(BODIES):
    got = answers["b%d" % index]["body"]
    if expected is None:
        response = json.loads(got)
        ok = list(response) == ["errors"] and len(response["errors"]) == 1 and \
            response["errors"][0]["message"].startswith("the request body is not JSON: ")
    else:
        ok = got == expected
    print("%-36s %s" % (label, got))
    differing += 0 if ok else 1
print("null fields equal absent ones:", answers["n0"]["body"] == answers["n1"]["body"])
differing += 0 if answers["n0"]["body"] == answers["n1"]["body"] else 1
DIFFS["bodies"] = differing

## The facade of a host

The sections above judge `aosabi_core::embed::answer` through the devtool. This section judges the
entry a host really links, `aosabi::embed::Embedded::boot(entry, aosabi::embed_output!())`, through
a program that holds it. The host `tests/fixtures/rs/embed-host` boots the app
`tests/fixtures/rs/embed-app` and answers one request body per line of its standard input. The app
holds a union, whose type resolution the build writes for the host, a field that resolves after a
timer, an error a resolver answers on purpose, an unexpected error, one that is no GraphQL error,
and a mutation, and beside them the fields `locale` and `count` of the section The key of the
token cap.

The cell builds the app with the card `server`, whose binary answers documents in the mode
`--requests`, then with the card `worker`, which builds the library of the app for
`wasm32-unknown-unknown` and writes the generated copy the host links, and then the host, every
build with `--locked`. The host answers the bodies in one process and the binary of the card answers
the documents of the requests among them in another, in the same order, so the mutation changes
both drawers alike. Each answer to a request is the result text of the binary byte for byte. Each
answer to a body that is no request is the answer of the operation `embed` of the devtool over the
table of the section above byte for byte, since such a body never reaches a service. A bad body sits
between two requests, so the host shows that it answers on. The cell then checks that
`{ __typename }` answers `{"data":{"__typename":"Query"}}` and that `--schema` prints the schema the
build wrote.

The app sets no key `maskedErrors` in its default, so the unexpected error answers its own message,
as it does over HTTP. The host then answers the same lines once more with the app in the variant
`masked_errors_true` of `AOSABI_CONFIG`, which turns the key on, a named divergence. Every answer is
then the one of the default but the answer to the unexpected error, which is its mask as an answer
over HTTP writes it, `Unexpected error.` with the code `INTERNAL_SERVER_ERROR` and the `locations`
and the `path` of the error. The error the resolver answers on purpose is a GraphQL error, which the
mask keeps, so its answer keeps its bytes. The engine of a host keeps the key in a form its threads
share, without a mask function of the app's own, so the variant `masked_errors_function`, which sets
one, fails the boot of the host with the exit status 70 and the error that says so on standard
error.

The app sets no key `introspection` in its default either, so the introspection query
`{__schema{queryType{name}}}` answers the schema, as the binary of the card answers it. The host
answers the same lines once more with the app in the variant `introspection_false`, a named
divergence. Every answer is then the one of the default but the answer to that query, which is the
body the HTTP road answers it with under the key: the errors of the refusal, each with the code
`GRAPHQL_VALIDATION_FAILED`. The cell holds that answer to its shape in every checkout, which is no
`data`, every error with the code `GRAPHQL_VALIDATION_FAILED` and a body other than the answer of
the default, and to a recording of the body byte for byte, an optional place of the notebook. A
checkout without the recording judges every answer and check of the section, the shape of the
refusal among them, prints that the bytes of the refusal need the recording, counts them as not
judged in the section and in the verdict, and does not fail.

In [ ]:
SERVED = APP / "target" / "debug" / "embed-app-server"
WORKER_LIBRARY = APP / "target" / "wasm32-unknown-unknown" / "debug" / "libembed_app.rlib"
HOST_TARGET = P.target / "embed-host"
HOST_BINARY = HOST_TARGET / "debug" / "embed-host"
FACADE = OUT / "facade"
FACADE.mkdir(parents=True, exist_ok=True)

# The three builds share the generated copy of the app, so they hold the lock
# of its directory, as the builds of the harness do. The host builds into a
# directory of its own under target/, as a probe of tests/fixtures/rs does.
BUILD_ENV = build_env(rustup_cargo())
BUILDS = [
    ("cargo aosabi build --locked, the card server",
     [CARGO_AOSABI, "build", "--locked", "--manifest-path", APP / "Cargo.toml"]),
    ("cargo aosabi build --locked --device worker",
     [CARGO_AOSABI, "build", "--locked", "--device", "worker", "--manifest-path", APP / "Cargo.toml"]),
    ("cargo build --locked, the host",
     [CARGO, "build", "--locked", "--manifest-path", HOST / "Cargo.toml", "--target-dir", HOST_TARGET]),
]
with lock_dir(APP):
    for label, argv in BUILDS:
        built = run(argv, cwd=P.root, env=BUILD_ENV, timeout_s=timeout_s, check=False)
        print("build %-46s exit %d, %.1f s" % (label, built.returncode, built.duration_s))
        if built.returncode != 0:
            print(built.report()[-4000:])
        assert built.returncode == 0, label
print("library for the Worker:", P.rel(WORKER_LIBRARY), WORKER_LIBRARY.is_file())
assert WORKER_LIBRARY.is_file(), WORKER_LIBRARY


def request_body(query, **fields):
    """Returns the JSON body a client sends over HTTP for `query` and `fields`."""
    return json.dumps({"query": query, **fields}, ensure_ascii=False)


UNION = "{ hello shapes { __typename ... on Circle { radius } ... on Square { side } } }"
# The introspection query, whose answer the key introspection decides.
INTROSPECTION_QUERY = "{__schema{queryType{name}}}"
TWO_OPERATIONS = "query Greet($name: String) { hello(name: $name) } query Other { __typename }"
# The lines of the host in their order, each with whether it is a request.
LINES = [
    ("the type name of the root", request_body("{ __typename }"), True),
    ("not json", "{ broken", False),
    ("a greeting and the union after a timer", request_body(UNION), True),
    ("an array", "[1]", False),
    ("variables and an operation name",
     request_body(TWO_OPERATIONS, variables={"name": "Ada"}, operationName="Greet"), True),
    ("no query", "{}", False),
    ("an error a resolver answers", request_body("{ shape(index: 7) { __typename } }"), True),
    ("a document that does not validate", request_body("{ nope }"), True),
    ("a mutation", request_body("mutation { addSquare(side: 0.5) }"), True),
    ("the union after the mutation", request_body(UNION), True),
    ("an introspection query", request_body(INTROSPECTION_QUERY), True),
    ("an unexpected error of a resolver", request_body("{ label }"), True),
]
# The line of the introspection query, whose answer the key introspection
# decides.
INTROSPECTION = len(LINES) - 2
# The line of the unexpected error, whose answer the key maskedErrors decides.
UNEXPECTED = len(LINES) - 1

# The host reads every line from one standard input, an empty line among them,
# which it skips.
host_input = "\n".join(text for _, text, _ in LINES[:3]) + "\n\n" + \
    "\n".join(text for _, text, _ in LINES[3:]) + "\n"
(FACADE / "host.in").write_text(host_input, encoding="utf-8")
# The app boots in its default, which sets no key maskedErrors.
DEFAULT_ENV = {"AOSABI_CONFIG": None}
hosted = run([HOST_BINARY], cwd=P.root, env=DEFAULT_ENV, stdin=host_input.encode("utf-8"),
             timeout_s=timeout_s)
(FACADE / "host.out").write_bytes(hosted.stdout)
host_answers = hosted.out.splitlines()
assert len(host_answers) == len(LINES), (len(host_answers), len(LINES))

# The served binary of the card answers the documents of the requests.
served_lines = []
for index, (_, text, is_request) in enumerate(LINES):
    if is_request:
        body = json.loads(text)
        line = {"id": "g%d" % index, "op": "execute", "document": body["query"]}
        for key in ("variables", "operationName"):
            if key in body:
                line[key] = body[key]
        served_lines.append(line)
served_path = FACADE / "served.ndjson"
served_path.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in served_lines), encoding="utf-8")
served = run([SERVED, "--requests", served_path], cwd=P.root, env=DEFAULT_ENV, timeout_s=timeout_s)
(FACADE / "served.answers.ndjson").write_bytes(served.stdout)
expected = {}
for line in served.out.splitlines():
    answer = json.loads(line)
    assert "result" in answer, answer
    expected[answer["id"]] = answer["result"]

# The devtool answers the bodies that are no request through the entry below.
bad = [{"id": "t", "op": "table", "table": table}]
for index, (_, text, is_request) in enumerate(LINES):
    if not is_request:
        bad.append({"id": "b%d" % index, "op": "embed", "table": name, "body": text})
for answer_id, answer in batch(bad, "facade-bodies").items():
    if answer_id != "t":
        expected[answer_id] = answer["body"]

differing = 0
for index, (label, _, is_request) in enumerate(LINES):
    key = ("g%d" if is_request else "b%d") % index
    side = "served binary" if is_request else "devtool embed"
    same = host_answers[index] == expected[key]
    print("%-40s %-14s %s  %s" % (label, side, "equal" if same else "DIFFERS", host_answers[index]))
    if not same:
        differing += 1
        bytediff(expected[key].encode(), host_answers[index].encode(), "facade %s" % label)

# The same lines once more with the app in the variant masked_errors_true,
# which turns the key maskedErrors on. Each answer is the one of the
# default but the answer to the unexpected error, which is its mask as an
# answer over HTTP writes it: Unexpected error. with the code
# INTERNAL_SERVER_ERROR and the locations and the path of the error, with no
# extension unexpected. The error the resolver answers on purpose is a GraphQL
# error, which the mask keeps.
masked_run = run([HOST_BINARY], cwd=P.root, env={"AOSABI_CONFIG": "masked_errors_true"},
                 stdin=host_input.encode("utf-8"), timeout_s=timeout_s)
(FACADE / "host.masked.out").write_bytes(masked_run.stdout)
masked_answers = masked_run.out.splitlines()
assert len(masked_answers) == len(LINES), (len(masked_answers), len(LINES))
unexpected = json.loads(host_answers[UNEXPECTED])
masked_error = dict(unexpected["errors"][0], message="Unexpected error.",
                    extensions={"code": "INTERNAL_SERVER_ERROR"})
masked_expected = json.dumps(dict(unexpected, errors=[masked_error]), ensure_ascii=False,
                             separators=(",", ":"))
masked_differing = 0
for index, (label, _, _) in enumerate(LINES):
    want = masked_expected if index == UNEXPECTED else host_answers[index]
    same = masked_answers[index] == want
    print("%-40s %-14s %s  %s" % (label, "the key on", "equal" if same else "DIFFERS", masked_answers[index]))
    if not same:
        masked_differing += 1
        bytediff(want.encode(), masked_answers[index].encode(), "facade masked %s" % label)

# The same lines once more with the app in the variant introspection_false,
# which turns introspection off. Each answer is the one of the default but the
# answer to the introspection query, which is the body the HTTP road answers
# that document with under the key: the errors of the refusal with the code
# GRAPHQL_VALIDATION_FAILED. A check below holds that answer to its shape in
# every checkout, and the recording of the body, an optional place, holds it
# to its bytes. Where the checkout holds no recording, golden_read raises
# GoldenMissing and the bytes of that one answer are not judged.
RECORDING = "capture/C50/introspection_off"
try:
    golden_request = golden_read(RECORDING + "/request.bin", root=P.root).split(b"\r\n\r\n", 1)[1]
    assert json.loads(golden_request) == json.loads(LINES[INTROSPECTION][1]), golden_request
    refused_body = golden_read(RECORDING + "/answer.bin", root=P.root).split(b"\r\n\r\n", 1)[1].decode("utf-8")
except GoldenMissing:
    refused_body = None
    NOT_JUDGED.append("the bytes of the answer to the introspection query with the key off, which need "
                      "the recording of that answer among the goldens")
    print("not judged: %s" % NOT_JUDGED[-1])
closed_run = run([HOST_BINARY], cwd=P.root, env={"AOSABI_CONFIG": "introspection_false"},
                 stdin=host_input.encode("utf-8"), timeout_s=timeout_s)
(FACADE / "host.introspection.out").write_bytes(closed_run.stdout)
closed_answers = closed_run.out.splitlines()
assert len(closed_answers) == len(LINES), (len(closed_answers), len(LINES))
closed_differing = 0
for index, (label, _, _) in enumerate(LINES):
    if index == INTROSPECTION and refused_body is None:
        print("%-40s %-14s %s  %s" % (label, "the key off", "by its shape", closed_answers[index]))
        continue
    want = refused_body if index == INTROSPECTION else host_answers[index]
    same = closed_answers[index] == want
    print("%-40s %-14s %s  %s" % (label, "the key off", "equal" if same else "DIFFERS", closed_answers[index]))
    if not same:
        closed_differing += 1
        bytediff(want.encode(), closed_answers[index].encode(), "facade introspection %s" % label)
if refused_body is None:
    print("the key off: %d of %d answers judged by their bytes, the refusal by its shape alone" % (
        len(LINES) - 1, len(LINES)))

# The variant masked_errors_function sets a mask function of the app's own in
# the key, which the embed road does not take, so the host fails its boot with
# 70 and the error of the boot on standard error.
refused = run([HOST_BINARY], cwd=P.root, env={"AOSABI_CONFIG": "masked_errors_function"},
              stdin=b"", timeout_s=timeout_s, check=False)
(FACADE / "host.function.err").write_bytes(refused.stderr)
MASK_FUNCTION = ("embed-host: The embed road takes the key maskedErrors without a mask function "
                 "of its own, and the key of the app sets mask_error. A card serves the app with "
                 "it over HTTP.")
print("the variant masked_errors_function exits %s: %s" % (refused.returncode, refused.err.strip()))

# The answers say what the app holds, so equal answers are no equal failures.
union_before = json.loads(host_answers[2])
union_after = json.loads(host_answers[9])
# The refusal under the key, which the checks hold to its shape in every
# checkout: no data, every error with the code of a validation failure, and
# a body other than the answer of the default.
refusal = json.loads(closed_answers[INTROSPECTION])
schema = run([HOST_BINARY, "--schema"], cwd=P.root, timeout_s=timeout_s).stdout
written = (APP / "target" / "aosabi" / "schema.graphql").read_bytes()
CHECKS = [
    ("{ __typename } answers the root", host_answers[0] == '{"data":{"__typename":"Query"}}'),
    ("the union resolves each member", union_before == {"data": {"hello": "Hello, world", "shapes": [
        {"__typename": "Circle", "radius": 1.5}, {"__typename": "Square", "side": 2}]}}),
    ("the variables reach the operation", json.loads(host_answers[4]) == {"data": {"hello": "Hello, Ada"}}),
    ("the error carries its code", json.loads(host_answers[6])["errors"][0]["extensions"]["code"] == "BAD_USER_INPUT"),
    ("the mutation changes the drawer", [s["__typename"] for s in union_after["data"]["shapes"]] ==
     ["Circle", "Square", "Square"]),
    ("--schema prints the schema of the build", schema == written),
    ("the default answers its own message", unexpected["errors"][0]["message"] ==
     "The store of labels is out of reach"),
    ("a mask function fails the boot", refused.returncode == 70 and
     MASK_FUNCTION in refused.err.splitlines()),
    ("the default answers the introspection", json.loads(host_answers[INTROSPECTION]) ==
     {"data": {"__schema": {"queryType": {"name": "Query"}}}}),
    ("the key off refuses the introspection", "data" not in refusal and bool(refusal.get("errors")) and
     all(error.get("extensions", {}).get("code") == "GRAPHQL_VALIDATION_FAILED" for error in refusal["errors"])
     and closed_answers[INTROSPECTION] != host_answers[INTROSPECTION]),
]
for label, ok in CHECKS:
    print("%-40s %s" % (label, "holds" if ok else "FAILS"))
DIFFS["facade"] = differing + masked_differing + closed_differing + sum(
    0 if ok else 1 for _, ok in CHECKS)

## A table without a service

The entry needs a service, and a table of the mode `plain` builds none, so the devtool refuses an
`embed` against it as a mistake of the harness, as it refuses a table the run does not hold.

In [ ]:
plain_table = json.loads((CORPUS / "blog-app-plain.json").read_text(encoding="utf-8"))["tables"][0]
answers = batch([
    {"id": "t", "op": "table", "table": plain_table},
    {"id": "p", "op": "embed", "table": plain_table["name"], "body": "{}"},
    {"id": "m", "op": "embed", "table": "no-such-table", "body": "{}"},
], "plain")
print(json.dumps(answers["p"]))
print(json.dumps(answers["m"]))
refusals_ok = (
    answers["p"].get("fatal") == "embed: the table %s is of the mode plain" % plain_table["name"]
    and answers["m"].get("fatal") == "embed: the run holds no table named no-such-table"
)
DIFFS["refusals"] = 0 if refusals_ok else 1

## The key of a fresh thread

The maps of `aosabi-json` whose builder is `aosabi_json::value::KeyState` hash their keys under the
key of the thread that makes them. On `wasm32-unknown-unknown`, `xous` and `vexos` that is every
map, and elsewhere it is the tables that name the state, among them the cache of parsed documents,
which takes the key of its thread at its first text once the thread holds one. A thread holds no key
until an entry of the engine stores one, and a host may hand its requests to threads of its own, so
each entry that answers a request keys the thread that calls it before it reads the request.

The operation `keys` of the devtool calls each entry on two threads that never reached the engine,
one after the other: `Service::execute`, `Service::subscribe`, the `Execute` of `Service` with a
query and with a subscription, `aosabi_core::request_stream::answer_requests`,
`aosabi_core::embed::answer` and `BootedApp::fetch` of an app with no routes. Each thread must hold
no key before the call and one after it. After the call each thread must hash a constant text to
another value than under the fixed key it held before, and the two threads must hold different
keys, so a seed of zeros or of any other constant fails. Both threads must answer the same. A
request line that names no operation of the protocol, a body that is no request and a fetch that no
route answers reach no service, so for them the entry alone keys the thread. A query must answer on
such a thread as on the thread of the run, the `Execute` of `Service` must answer a subscription as
`Service::subscribe` does, and an entry of no such name is a mistake of the harness.

On the host of the build an entry keys the thread with no step of the host. The probe runs there,
where the maps that the reader of a body makes hash under the keys of the standard library, so it
cannot see whether an entry keys its thread before or after it parses the body. On the three targets
above that order decides the key of those maps, and each entry keys its thread in its first
statement. A host on those targets, whose standard library has no source of random bytes of its own,
installs a provider before its first request, as the doc of `aosabi::embed` says. Until it does,
every entry above answers a refusal in place of the request, which the section A target without
random bytes judges on such a target for the facade. The middleware of `aosabi_core::app::handler`
and the fetch of `aosabi_core::graphtp::served` key their threads and refuse in the same way, and
the probe reaches neither.

In [ ]:
# Each entry on two fresh threads, with the query and the table of the section,
# a subscription the schema has no root for, bodies that are no request and a
# fetch that no route answers. Each check takes the answer of the entry, the
# result of the same query on the thread of the run, and the answers of the
# calls before it by entry and label.
subscription = "subscription { __typename }"


def errors_only(text):
    """Whether `text` is a JSON response that carries errors and no data."""
    try:
        response = json.loads(text)
    except ValueError:
        return False
    return isinstance(response, dict) and bool(response.get("errors")) and response.get("data") is None


KEY_CALLS = [
    ("execute", "the query", "document", query, lambda text, run, got: text == run),
    ("subscribe", "the query", "document", query, lambda text, run, got: text == run),
    ("subscribe", "a subscription", "document", subscription, lambda text, run, got: errors_only(text)),
    ("trait", "the query", "document", query, lambda text, run, got: text == run),
    ("trait", "a subscription", "document", subscription,
     lambda text, run, got: text == got.get(("subscribe", "a subscription"))),
    ("requests", "a line of the query", "requests", json.dumps({"id": "r", "op": "execute", "document": query}) + "\n",
     lambda text, run, got: json.loads(text) == {"id": "r", "op": "execute", "result": run}),
    ("requests", "a line of no operation", "requests", json.dumps({"id": "u", "op": "no-such-op"}) + "\n",
     lambda text, run, got: sorted(json.loads(text)) == ["fatal", "id", "op"]),
    ("embed", "the body of the query", "body", json.dumps({"query": query}), lambda text, run, got: text == run),
    ("embed", "a body that is no request", "body", "{}",
     lambda text, run, got: text == failure("the request carries no string field query")),
    ("fetch", "a path no route answers", "url", "http://localhost/graphql", lambda text, run, got: text == "404"),
]
KEYED = {"before": False, "after": True, "moved": True, "distinct": True, "agree": True}
requests = [
    {"id": "t", "op": "table", "table": table},
    {"id": "x", "op": "execute", "table": name, "document": query},
]
for index, (entry, _, field, value, _) in enumerate(KEY_CALLS):
    requests.append({"id": "k%d" % index, "op": "keys", "table": name, "entry": entry, field: value})
requests.append({"id": "e", "op": "keys", "table": name, "entry": "no-such-entry", "document": query})
answers = batch(requests, "keys")
assert answers["t"].get("ok") is True, answers["t"]
on_the_run = answers["x"]["result"]
differing = 0
got = {}
for index, (entry, label, _, _, check) in enumerate(KEY_CALLS):
    answer = answers["k%d" % index]
    keyed, text = answer.get("keyed") or {}, answer.get("answer")
    ok = keyed == KEYED and text is not None and check(text, on_the_run, got)
    got[(entry, label)] = text
    print("%-9s %-26s %s, %s" % (
        entry, label, " ".join("%s %s" % (key, keyed.get(key)) for key in KEYED),
        "the answer it must give" if ok else "answered %s" % json.dumps(answer)))
    differing += 0 if ok else 1
print(json.dumps(answers["e"]))
differing += 0 if answers["e"].get("fatal") == "keys: no entry named no-such-entry" else 1
DIFFS["keys"] = differing

## A target without random bytes

On `wasm32-unknown-unknown`, `xous` and `vexos` the standard library has no source of random bytes,
so a thread there holds a key only once a provider fills one, which the host installs with
`install_randomness`. Until then `aosabi_core::embed::answer`, the entry of `aosabi::embed`, answers
every body with the response of one error and no data, and reads no part of it:
`{"errors":[{"message":"no source of random bytes: the engine answers no request on this target until the host installs a provider with install_randomness"}]}`.
`Service::execute`, `Service::subscribe`, the `Execute` of `Service` and the request lines answer
the same message in place of a request.

The program `tests/fixtures/rs/embed-wasm` boots the app of the section The facade of a host through
the same facade on `wasm32-unknown-unknown`, over the generated copy that section wrote for the card
`worker`. It hands the app `{ __typename }` and the body `{ broken`, which is no JSON, installs the
provider of the process with `aosabi::embed::install_randomness`, and hands it `{ __typename }`
again. The module imports nothing, so its provider hands out the count of the bytes it gave. The
target has no standard output, so the status of `main` is the verdict, one bit per check that fails:
1 when the request before the install answers another body than the refusal, 2 when the body that is
no JSON does, 4 when the thread holds a key before the install, 8 when the request after the install
answers another body than `{"data":{"__typename":"Query"}}`, 16 when the thread holds no key after
it, 32 when the provider handed out another count of bytes than the sixteen of one key, and 128
alone when the boot fails. The cell builds the program with `--locked` under the lock of the app and
runs its `main` on Node with an empty import object, so the module must import nothing and answer
the status 0.

In [ ]:
# The host of the facade on wasm32-unknown-unknown, with the bits of the status
# of its main as tests/fixtures/rs/embed-wasm/src/main.rs names them.
WASM_TARGET = P.target / "embed-wasm"
WASM_MODULE = WASM_TARGET / "wasm32-unknown-unknown" / "debug" / "embed-wasm.wasm"
WASM_CHECKS = [
    (1, "a request before the install answers the refusal"),
    (2, "a body that is no JSON answers the refusal"),
    (4, "the thread holds no key before the install"),
    (8, "the request answers after the install"),
    (16, "the thread holds a key after the install"),
    (32, "the provider handed out the sixteen bytes of one key"),
]
WASM_BOOT = 128

with lock_dir(APP):
    built = run([CARGO, "build", "--locked", "--target", "wasm32-unknown-unknown",
                 "--manifest-path", WASM_HOST / "Cargo.toml", "--target-dir", WASM_TARGET],
                cwd=P.root, env=BUILD_ENV, timeout_s=timeout_s, check=False)
print("build %-46s exit %d, %.1f s" % ("cargo build --locked, the host on wasm32", built.returncode,
                                       built.duration_s))
if built.returncode != 0:
    print(built.report()[-4000:])
assert built.returncode == 0

# Node instantiates the module with an empty import object and calls the main
# the target exports with no argument.
RUN_MAIN = r"""
const { readFileSync } = await import("node:fs");
const module = new WebAssembly.Module(readFileSync(process.argv[1]));
const imports = WebAssembly.Module.imports(module).map((entry) => entry.module + "." + entry.name);
const names = WebAssembly.Module.exports(module).map((entry) => entry.name);
const main = names.includes("__main_argc_argv") ? "__main_argc_argv" : "main";
const instance = new WebAssembly.Instance(module, {});
console.log(JSON.stringify({ imports, main, status: instance.exports[main](0, 0) }));
"""
ran = run(["node", "--input-type=module", "-e", RUN_MAIN, WASM_MODULE], cwd=P.root, timeout_s=timeout_s)
verdict = json.loads(ran.out)
status = verdict["status"]
print("module :", P.rel(WASM_MODULE), "%d bytes" % WASM_MODULE.stat().st_size)
print("imports:", verdict["imports"], " main:", verdict["main"], " status:", status)
if status & WASM_BOOT:
    print("the boot fails")
for bit, label in WASM_CHECKS:
    print("%-56s %s" % (label, "FAILS" if status & bit else "holds"))
DIFFS["no random bytes"] = 0 if verdict["imports"] == [] and status == 0 else 1

## The nesting bound of a request

A served engine parses every request document under a bound of 1,000 levels of nesting, the constant
`REQUEST_NESTING` of `crates/aosabi-lang/src/parser.rs`, and the entry answers through the same
service, so it holds the same bound. A selection set, a list value, an object value and a list type
of 1,000 levels each parse, so the first error of each comes from validation, since none of them
names a field or a type of the schema. Each of 1,001, 10,000 and 100,000 levels answers the syntax
error of the bound, `Document nests deeper than 1000 levels. Parsing aborted.`, as one error with no
data. The entry answers `{__typename}` after each of them.

In [ ]:
# The bound of a request document, REQUEST_NESTING of crates/aosabi-lang/src/parser.rs.
NESTING = 1000
NESTED_ERROR = "Syntax Error: Document nests deeper than %d levels. Parsing aborted." % NESTING


def nested(shape, depth):
    """The document of `shape` nested `depth` levels deep."""
    if shape == "selection set":
        return "{a" * depth + "}" * depth
    if shape == "list value":
        return "query($v: Int = " + "[" * depth + "1" + "]" * depth + ") { a }"
    if shape == "object value":
        return "query($v: Int = " + "{a:" * depth + "1" + "}" * depth + ") { a }"
    return "query($v: " + "[" * depth + "Int" + "]" * depth + ") { a }"


SHAPES = ("selection set", "list value", "object value", "list type")
DEPTHS = (NESTING, NESTING + 1, 10 * NESTING, 100 * NESTING)
table = json.loads((CORPUS / "blog-app-pylon.json").read_text(encoding="utf-8"))["tables"][0]
name = table["name"]
typename_body = json.dumps({"query": "{__typename}"})
requests = [{"id": "t", "op": "table", "table": table},
            {"id": "first", "op": "embed", "table": name, "body": typename_body}]
for s, shape in enumerate(SHAPES):
    for d, depth in enumerate(DEPTHS):
        requests.append({"id": "d%d.%d" % (s, d), "op": "embed", "table": name,
                         "body": json.dumps({"query": nested(shape, depth)})})
        requests.append({"id": "a%d.%d" % (s, d), "op": "embed", "table": name, "body": typename_body})
answers = batch(requests, "nesting")
assert answers["t"].get("ok") is True, answers["t"]
typename = answers["first"]["body"]
print("{__typename} answers", typename)
differing = 0
print("%-14s %7s  %s" % ("shape", "levels", "the first error of the answer"))
for s, shape in enumerate(SHAPES):
    for d, depth in enumerate(DEPTHS):
        response = json.loads(answers["d%d.%d" % (s, d)]["body"])
        errors = response.get("errors") or [{}]
        message = errors[0].get("message")
        print("%-14s %7s  %s" % (shape, "{:,}".format(depth), message))
        if depth <= NESTING:
            # A document at the bound parses, so its first error comes from validation.
            ok = message is not None and not message.startswith("Syntax Error") and "data" not in response
        else:
            ok = message == NESTED_ERROR and "data" not in response and len(errors) == 1
        if answers["a%d.%d" % (s, d)]["body"] != typename:
            ok = False
            print("  {__typename} after it answers", answers["a%d.%d" % (s, d)]["body"])
        differing += 0 if ok else 1
DIFFS["nesting"] = differing

## The token cap of a request

A served engine parses every request document under a cap of 120,008 tokens, the constant
`REQUEST_TOKENS` of `crates/aosabi-lang/src/parser.rs`, a named divergence, and the entry answers
through the same service, so it holds the same cap. Each document of the section declares a variable
it never uses, whose default value is a list of empty lists, so its parse is linear and validation
rejects it, `listed_document` of `tests/harness/divergences.py`. A document of 120,008 tokens
parses, so its first error comes from validation. Each of 120,009 and 1,200,080 tokens answers the
syntax error the parser of graphql-js answers under a budget of 120,008 tokens, which the cell reads
from `tests/sidecar/graphql-js.mjs`, graphql 16.12.0, under the options `request_parse_options` of
`tests/harness/divergences.py`, as one error with its location and no data. The output names that
error by its place and never by its text. The entry answers `{__typename}` after each of them.

In [ ]:
# The cap of a request document, REQUEST_TOKENS of crates/aosabi-lang/src/parser.rs.
TOKENS = 120008
from harness import divergences
from harness.divergences import listed_document

assert TOKENS == divergences.REQUEST_TOKENS, (TOKENS, divergences.REQUEST_TOKENS)
SIZES = (TOKENS, TOKENS + 1, 10 * TOKENS)
documents = {size: listed_document(size) for size in SIZES}

# The error of a document past the cap is the error the parser of graphql-js
# answers under a budget of the cap, read from the sidecar under the options
# `divergences.request_parse_options`.
SIDECAR = P.tests / "sidecar" / "graphql-js.mjs"
payload = "".join(json.dumps({"id": str(size), "op": "parse", "document": documents[size],
                              "options": divergences.request_parse_options()}) + "\n"
                  for size in SIZES if size > TOKENS).encode("utf-8")
parsed = {}
for line in run(["node", SIDECAR], cwd=P.root, stdin=payload, timeout_s=timeout_s).out.splitlines():
    answer = json.loads(line)
    parsed[int(answer["id"])] = answer

table = json.loads((CORPUS / "blog-app-pylon.json").read_text(encoding="utf-8"))["tables"][0]
name = table["name"]
typename_body = json.dumps({"query": "{__typename}"})
requests = [{"id": "t", "op": "table", "table": table},
            {"id": "first", "op": "embed", "table": name, "body": typename_body}]
for size in SIZES:
    requests.append({"id": "c%d" % size, "op": "embed", "table": name, "body": json.dumps({"query": documents[size]})})
    requests.append({"id": "a%d" % size, "op": "embed", "table": name, "body": typename_body})
answers = batch(requests, "token-cap")
assert answers["t"].get("ok") is True, answers["t"]
typename = answers["first"]["body"]
print("{__typename} answers", typename)
differing = 0
print("%9s  %s" % ("tokens", "the answer"))
for size in SIZES:
    response = json.loads(answers["c%d" % size]["body"])
    errors = response.get("errors") or [{}]
    message = errors[0].get("message")
    if size <= TOKENS:
        # A document at the cap parses, so its first error comes from validation.
        ok = message is not None and not message.startswith("Syntax Error") and "data" not in response
        print("%9s  %s" % ("{:,}".format(size), message))
    else:
        expected = (parsed.get(size, {}).get("errors") or [{}])[0]
        ok = (len(errors) == 1 and "data" not in response and expected.get("message") is not None
              and message == expected.get("message") and errors[0].get("locations") == expected.get("locations"))
        if ok:
            print("%9s  the error of the parse under a budget of %s at %s" % (
                "{:,}".format(size), "{:,}".format(TOKENS), errors[0].get("locations")))
        else:
            print("%9s  %d errors and %s, not the error of the parse under a budget of %s" % (
                "{:,}".format(size), len(errors), "data" if "data" in response else "no data", "{:,}".format(TOKENS)))
    if answers["a%d" % size]["body"] != typename:
        ok = False
        print("  {__typename} after it answers", answers["a%d" % size]["body"])
    differing += 0 if ok else 1
DIFFS["token cap"] = differing

## The key of the token cap

The key `maxTokens` of an app, `.max_tokens(..)` in its chain, raises or lowers the cap for every
parse of a request document of the app: the parse of the executor and the second parses of the
service, the one that asks whether the operation is a subscription and the one that reads
`@inContext`. The app `embed-app` reads the variant `AOSABI_CONFIG` names in its entry, as a twin of
the harness does: `tokens_low` sets the key to 1,000 and `tokens_high` to 240,016, twice the cap of
an app that sets none. Its field `locale` answers the locale the `@inContext` of the operation
names, and its field `count` the count of the list it is handed, so `counted_document` below writes
a document of any count of tokens that parses and validates and reads its locale.

Under each variant the host of the section The facade of a host and the binary of the card `server`
in the mode `--requests` answer, in one process each, documents of `listed_document` and of
`counted_document`, each followed by `{__typename}`: under `tokens_low` a listed and a counted
document of 1,000 and of 1,001 tokens, under `tokens_high` a listed document of 120,009, a counted
one of 120,009 and of 240,016 and a listed and a counted one of 240,017 tokens. Each answer of the
host must equal the result text of the binary byte for byte. A listed document within the cap of the
key must answer the error of its validation with no data, a counted one within it must answer its
count and the locale `de` of its `@inContext`, which the second parse reads under the cap of the
key, and a document past the cap of the key must answer the syntax error the parser of graphql-js
answers under a budget of that cap, read from `tests/sidecar/graphql-js.mjs` under the options
`request_parse_options` of `tests/harness/divergences.py` with that cap, as one error with its
location and no data. The output names that error by its place and never by its text.

In [ ]:
# The key maxTokens of the app on the facade of a host and in the mode --requests.
from harness.divergences import count_tokens

# The cap each variant of embed-app sets with the key.
KEY_CAPS = {"tokens_low": 1000, "tokens_high": 2 * TOKENS}
COUNTED = 'query @inContext(locale: "de") { count(items: [%s]) locale }'


def counted_document(tokens):
    """Returns a document of `tokens` tokens that parses and validates, whose operation counts a list of
    ones and reads the locale its @inContext names."""
    ones = tokens - count_tokens(COUNTED % "")
    document = COUNTED % " ".join(["1"] * ones)
    assert count_tokens(document) == tokens, (tokens, count_tokens(document))
    return document


KEY_CASES = {
    "tokens_low": [("listed", 1000), ("listed", 1001), ("counted", 1000), ("counted", 1001)],
    "tokens_high": [("listed", TOKENS + 1), ("counted", TOKENS + 1), ("counted", 2 * TOKENS),
                    ("listed", 2 * TOKENS + 1), ("counted", 2 * TOKENS + 1)],
}
KEY_DIR = OUT / "key"
KEY_DIR.mkdir(parents=True, exist_ok=True)
TYPENAME_ANSWER = '{"data":{"__typename":"Query"}}'
key_differing = 0
for variant, cap in KEY_CAPS.items():
    documents = {}
    for kind, size in KEY_CASES[variant]:
        text = listed_document(size) if kind == "listed" else counted_document(size)
        documents["%s document of %s tokens" % (kind, "{:,}".format(size))] = (kind, size, text)
    # The error of the parse of graphql-js under the cap of the key, for each document past it.
    payload = "".join(json.dumps({"id": name, "op": "parse", "document": text,
                                  "options": divergences.request_parse_options(max_tokens=cap)}) + "\n"
                      for name, (kind, size, text) in documents.items() if size > cap).encode("utf-8")
    key_parsed = {}
    for line in run(["node", SIDECAR], cwd=P.root, stdin=payload, timeout_s=timeout_s).out.splitlines():
        answer = json.loads(line)
        key_parsed[answer["id"]] = (answer.get("errors") or [{}])[0]
    # The host answers every document and {__typename} after it, the binary of the card the same lines.
    names = list(documents)
    lines = []
    for name in names:
        lines.append(json.dumps({"query": documents[name][2]}))
        lines.append(json.dumps({"query": "{__typename}"}))
    host_input = "\n".join(lines) + "\n"
    hosted = run([HOST_BINARY], cwd=P.root, env={"AOSABI_CONFIG": variant}, stdin=host_input.encode("utf-8"),
                 timeout_s=timeout_s)
    host_answers = hosted.out.splitlines()
    served_lines = [{"id": "k%d" % index, "op": "execute", "document": json.loads(line)["query"]}
                    for index, line in enumerate(lines)]
    served_path = KEY_DIR / ("%s.ndjson" % variant)
    served_path.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in served_lines), encoding="utf-8")
    served = run([SERVED, "--requests", served_path], cwd=P.root, env={"AOSABI_CONFIG": variant}, timeout_s=timeout_s)
    served_results = {}
    for line in served.out.splitlines():
        answer = json.loads(line)
        served_results[answer["id"]] = answer.get("result")
    print("%s, the key maxTokens at %s: %d answers of the host, %d of the binary" % (
        variant, "{:,}".format(cap), len(host_answers), len(served_results)))
    if len(host_answers) != len(lines) or len(served_results) != len(lines):
        key_differing += 1
        print("   the host or the binary answered another count of lines than %d" % len(lines))
        continue
    for index, name in enumerate(names):
        kind, size, text = documents[name]
        ours = host_answers[2 * index]
        ok = ours == served_results["k%d" % (2 * index)]
        response = json.loads(ours)
        errors = response.get("errors") or [{}]
        if size > cap:
            expected = key_parsed.get(name, {})
            held = (len(errors) == 1 and "data" not in response and expected.get("message") is not None
                    and errors[0].get("message") == expected.get("message")
                    and errors[0].get("locations") == expected.get("locations"))
            shown = "the error of the parse under a budget of %s at %s" % (
                "{:,}".format(cap), errors[0].get("locations")) if held else "%d errors and %s" % (
                len(errors), "data" if "data" in response else "no data")
        elif kind == "listed":
            message = errors[0].get("message")
            held = message is not None and not message.startswith("Syntax Error") and "data" not in response
            shown = "an error of its validation" if held else "%d errors and %s" % (
                len(errors), "data" if "data" in response else "no data")
        else:
            want = {"data": {"count": size - count_tokens(COUNTED % ""), "locale": "de"}}
            held = response == want
            shown = "its count and the locale de" if held else ours[:120]
        print("   %-36s %s, %s" % (name, shown, "equal to the binary" if ok else "NOT the result of the binary"))
        after_host = host_answers[2 * index + 1]
        after_served = served_results["k%d" % (2 * index + 1)]
        if after_host != TYPENAME_ANSWER or after_served != TYPENAME_ANSWER:
            held = False
            print("   {__typename} after it answers %s on the host and %s on the binary" % (after_host, after_served))
        if not ok:
            bytediff(served_results["k%d" % (2 * index)].encode(), ours.encode(), "key %s %s" % (variant, name))
        key_differing += 0 if ok and held else 1
DIFFS["key of the token cap"] = key_differing

## Verdict

In [ ]:
print("differences by section:", DIFFS)
print("cases compared:", compared)
if NOT_JUDGED:
    print("cases not judged: %d, %s" % (len(NOT_JUDGED), ", ".join(NOT_JUDGED)))
assert sum(DIFFS.values()) == 0, DIFFS
assert compared > 0